YOLOv8 Object Detection & Tracking Pipeline
Lab Implementation - Ultralytics YOLOv8

1. Frame Extraction → OpenCV reads video frame-by-frame
2. Detection → YOLOv8 detects objects
3. Tracking → ByteTrack assigns IDs
4. Annotation → Bounding boxes + IDs drawn
5. Output → Saved video

In [3]:
import cv2
import argparse
from ultralytics import YOLO

# CONFIG  (edit these or use CLI args)
DEFAULT_MODEL   = "yolov8n.pt"       
DEFAULT_INPUT   = "input.mp4"        # path to video, or 0 for webcam
DEFAULT_OUTPUT  = "output_tracked.mp4"
DEFAULT_CONF    = 0.5                # confidence threshold
DEFAULT_TRACKER = "bytetrack.yaml"   # or "botsort.yaml"

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\prady\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# PIPELINE FUNCTION

def run_pipeline(model_path, input_source, output_path,
                 conf_thresh, tracker_cfg, show=True):
    """
    Full YOLOv8 detect + track pipeline.

    Stages:
    1. Load YOLOv8 model
    2. Open video source
    3. Per-frame: detect → track → annotate
    4. Write annotated frames to output video
    """

    # ── Stage 1: Load model ─────────────────
    print(f"[INFO] Loading model: {model_path}")
    model = YOLO(model_path)

    # ── Stage 2: Open video ─────────────────
    cap = cv2.VideoCapture(input_source)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video source: {input_source}")

    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] Video: {W}x{H} @ {fps}fps  |  Frames: {total_frames}")

    # ── Video writer ─────────────────────────
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    frame_idx = 0

    # ── Stage 3 & 4: Main loop ───────────────
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # ── Detect + Track (single call) ──────
        # persist=True keeps tracker state between frames
        results = model.track(
            source=frame,
            conf=conf_thresh,
            tracker=tracker_cfg,
            persist=True,
            verbose=False,
        )

        # ── Annotate frame ────────────────────
        # results[0].plot() draws boxes, labels, track IDs
        annotated_frame = results[0].plot()

        # ── Write & display ───────────────────
        writer.write(annotated_frame)

        if show:
            cv2.imshow("YOLOv8 Object Detection + Tracking", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                print("[INFO] Quit key pressed — stopping early.")
                break

        # Progress log every 30 frames
        if frame_idx % 30 == 0:
            print(f"[INFO] Processed frame {frame_idx}/{total_frames}")

    # ── Cleanup ───────────────────────────────
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print(f"[DONE] Output saved to: {output_path}")
    print(f"[DONE] Total frames processed: {frame_idx}")

In [5]:
# MAIN

def parse_args():
    parser = argparse.ArgumentParser(
        description="YOLOv8 Video Object Detection & Tracking Pipeline"
    )
    parser.add_argument("--model",   default=DEFAULT_MODEL,
                        help="YOLOv8 model weights (yolov8n/s/m/l/x.pt)")
    parser.add_argument("--input",   default=DEFAULT_INPUT,
                        help="Input video path or webcam index (0)")
    parser.add_argument("--output",  default=DEFAULT_OUTPUT,
                        help="Output video path")
    parser.add_argument("--conf",    type=float, default=DEFAULT_CONF,
                        help="Confidence threshold (0–1)")
    parser.add_argument("--tracker", default=DEFAULT_TRACKER,
                        help="Tracker config: bytetrack.yaml or botsort.yaml")
    parser.add_argument("--no-show", action="store_true",
                        help="Disable live preview window")
    return parser.parse_args()

In [7]:
if __name__ == "__main__":
    args = parse_args()

    # Convert webcam index string to int
    source = int(args.input) if args.input.isdigit() else args.input

    run_pipeline(
        model_path   = args.model,
        input_source = source,
        output_path  = args.output,
        conf_thresh  = args.conf,
        tracker_cfg  = args.tracker,
        show         = not args.no_show,
    )

usage: ipykernel_launcher.py [-h] [--model MODEL] [--input INPUT] [--output OUTPUT] [--conf CONF] [--tracker TRACKER]
                             [--no-show]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\prady\AppData\Roaming\jupyter\runtime\kernel-0c3be8e7-d0f2-4fe2-b88b-b9eab6359df3.json


SystemExit: 2